### Instalación de Librerías
Instalamos `fasttext`, una librería creada por Facebook AI para el aprendizaje eficiente de representaciones de palabras (word embeddings) y clasificación de texto.

In [8]:
# Instalación de la librería (si no está instalada)
!pip install fasttext

In [9]:
import fasttext
import os

### Obtención del Corpus de Texto
Para entrenar nuestro propio modelo de lenguaje, necesitamos un texto base. Descargamos **"Don Quijote de la Mancha"** desde el Proyecto Gutenberg. Este será el "universo" del que nuestro modelo aprenderá el significado de las palabras según su contexto.

In [10]:
# Descarga de datos (Don Quijote desde Project Gutenberg)
print("Descargando el texto de El Quijote...")
!wget -O quijote.txt https://www.gutenberg.org/cache/epub/2000/pg2000.txt

# Verificamos que se descargó
if os.path.exists('quijote.txt'):
    print("¡Archivo descargado correctamente: 'quijote.txt'!")
else:
    print("Error en la descarga.")

Descargando el texto de El Quijote...
--2026-01-21 18:55:06--  https://www.gutenberg.org/cache/epub/2000/pg2000.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2225845 (2.1M) [text/plain]
Saving to: ‘quijote.txt’

quijote.txt         100%[===================>]   2.12M  8.45MB/s    in 0.3s    

2026-01-21 18:55:06 (8.45 MB/s) - ‘quijote.txt’ saved [2225845/2225845]

¡Archivo descargado correctamente: 'quijote.txt'!


### Entrenamiento del Modelo de Embeddings
Entrenamos el modelo usando el algoritmo **Skipgram** (que predice el contexto dado una palabra).
* `minCount=5`: Ignoramos palabras muy poco frecuentes (posibles erratas).
* `epoch=10`: El modelo leerá el libro entero 10 veces para ajustar bien los vectores.

In [11]:
# Entrenamiento del modelo
# Ajustamos los parámetros:
# minCount=5: Ignoramos palabras que aparecen menos de 5 veces (elimina ruido/erratas)
# epoch=10: Como el texto es largo, 10 pasadas son suficientes
print("Entrenando modelo (esto puede tardar unos segundos)...")
model = fasttext.train_unsupervised('quijote.txt', model='skipgram', minCount=5, epoch=10)

Entrenando modelo (esto puede tardar unos segundos)...


### Inspección del Vector
Aquí cumplimos dos objetivos:
1.  **Visualizar el Vector:** Vemos cómo el modelo ha transformado la palabra "caballero" en una lista de números (el embedding) que captura su significado matemático.
2.  **Verificar Vecinos:** Comprobamos qué otras palabras están cerca en este espacio vectorial.

In [12]:
# Interrogar al modelo
palabra_test = "caballero"

# Visualizamos el vector numérico de la palabra
if palabra_test in model.words:
    vector = model[palabra_test]
    print(f"Vector de la palabra '{palabra_test}' (primeros 20 valores):")
    print(vector[:20]) # Mostramos solo el principio para no llenar la pantalla
    print(f"Longitud total del vector: {len(vector)}\n")

# Verificamos si la palabra está en el vocabulario del modelo
if palabra_test in model.words:
    print(f"\nBuscando vecinos para: '{palabra_test}'")
    vecinos = model.get_nearest_neighbors(palabra_test, k=5)

    for similitud, palabra in vecinos:
        print(f"- {palabra} (Similitud: {similitud:.4f})")
else:
    print(f"La palabra '{palabra_test}' no aparece suficientes veces en el texto.")

Vector de la palabra 'caballero' (primeros 20 valores):
[-0.19331773  0.2598694  -0.7667309  -0.72955227  0.1832145  -0.03884666
 -0.50244695  0.03627121 -0.20301747  0.6451019  -0.31212768 -0.4699186
 -0.19616966 -0.61947894 -0.652842    0.94615227 -0.97198063 -0.382032
  0.60746986  0.5309442 ]
Longitud total del vector: 100


Buscando vecinos para: 'caballero'
- caballero. (Similitud: 0.9817)
- caballero; (Similitud: 0.9798)
- caballero? (Similitud: 0.9754)
- caballeriza (Similitud: 0.9603)
- caballero, (Similitud: 0.9599)


### Interfaz de Búsqueda Interactiva
Utilizamos `model.get_nearest_neighbors` conectado a un `input()` de teclado.
Esto nos permite probar cualquier palabra en tiempo real y ver qué asociaciones semánticas ha aprendido el modelo (por ejemplo, ver qué palabras asocia con "escudo" o "sancho").

In [14]:
# Interrogatorio interactivo
# Usamos input() para pedir la palabra por teclado
palabra_usuario = input("Introduce una palabra para buscar sus relacionadas (ej. ej. sancho, escudo, dios): ")

# Buscamos las 5 palabras más cercanas (k=5)
# FastText devuelve una lista de tuplas (similitud, palabra)
# get_nearest_neighbors devuelve primero la probabilidad, luego la palabra.
vecinos = model.get_nearest_neighbors(palabra_usuario, k=5)

print(f"\nPalabras más relacionadas con '{palabra_usuario}':")
for similitud, palabra in vecinos:
    print(f"- {palabra} (Similitud: {similitud:.4f})")

Introduce una palabra para buscar sus relacionadas (ej. ej. sancho, escudo, dios): gigantes

Palabras más relacionadas con 'gigantes':
- gigantes, (Similitud: 0.9070)
- andantes (Similitud: 0.8533)
- amantes (Similitud: 0.8434)
- tocantes (Similitud: 0.8327)
- andantes. (Similitud: 0.8282)
